# Weekly Population Interpolation

This notebook builds a linear, integer-only population trajectory for each census sector using the epidemic weeks present in the processed dengue data. It also compares the raw endpoints and the interpolated series so the DVC stage is easy to inspect.

In [ ]:
import os
os.listdir(PROJECT_ROOT)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'params.yaml').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
sys.path.append('../../')

from src.census_equivalence_2010_2022 import (
    build_extrapolated_population_table,
    load_epidemic_weeks_from_dengue,
)

def melt_population_table(population_table: pd.DataFrame) -> pd.DataFrame:
    week_columns = [
        column
        for column in population_table.columns
        if column not in {'sector_id', 'population_2010', 'population_2022'}
    ]
    melted_table = population_table.melt(
        id_vars=['sector_id', 'population_2010', 'population_2022'],
        value_vars=week_columns,
        var_name='epidemic_date',
        value_name='interpolated_population',
    )
    melted_table['week_index'] = melted_table['epidemic_date'].map(
        {week: index for index, week in enumerate(week_columns)}
    )
    return melted_table.sort_values(['sector_id', 'week_index']).reset_index(drop=True)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

## Load data and build the interpolation

In [ ]:
with open(PROJECT_ROOT / 'params.yaml', 'r', encoding='utf-8') as file_handle:
    params = yaml.safe_load(file_handle)

population_path = PROJECT_ROOT / params['all']['paths']['data']['processed']['census_equivalence']['population_2010_to_2022']
dengue_path = PROJECT_ROOT / params['all']['paths']['data']['processed']['dengue']
output_path = PROJECT_ROOT / params['all']['paths']['data']['processed']['population_interpolated']

population_data = pd.read_csv(population_path, dtype={'sector_id': str})
epidemic_weeks = load_epidemic_weeks_from_dengue(dengue_path)
interpolated_population = build_extrapolated_population_table(population_data, epidemic_weeks)
interpolated_population.to_csv(output_path, index=False)

print(f'Population input: {population_path}')
print(f'Dengue input: {dengue_path}')
print(f'Interpolated output: {output_path}')
print(f'Sectors: {len(population_data):,}')
print(f'Epidemic weeks: {len(epidemic_weeks):,}')
display(population_data.head())
display(interpolated_population.iloc[:, :8].head())

## Summary of the interpolation

In [ ]:
population_change = population_data['population_2022'] - population_data['population_2010']
summary = pd.DataFrame(
    {
        'metric': ['min', 'median', 'mean', 'max'],
        'population_change': [
            population_change.min(),
            population_change.median(),
            population_change.mean(),
            population_change.max(),
        ],
    }
)
display(summary)
print('Rows with a population increase:', int((population_change > 0).sum()))
print('Rows with a population decrease:', int((population_change < 0).sum()))
print('Rows without change:', int((population_change == 0).sum()))

## Sample sector trajectories

In [ ]:
population_change = population_data['population_2022'] - population_data['population_2010']
sample_sector_ids = [
    population_data.loc[population_change.idxmax(), 'sector_id'],
    population_data.loc[population_change.idxmin(), 'sector_id'],
    population_data.iloc[len(population_data) // 2]['sector_id'],
]
sample_table = interpolated_population[interpolated_population['sector_id'].isin(sample_sector_ids)].copy()
long_sample = melt_population_table(sample_table)
week_labels = [epidemic_weeks[0], epidemic_weeks[len(epidemic_weeks) // 2], epidemic_weeks[-1]]
week_positions = [0, len(epidemic_weeks) // 2, len(epidemic_weeks) - 1]

fig, ax = plt.subplots(figsize=(14, 6))
for sector_id, group in long_sample.groupby('sector_id'):
    ax.plot(group['week_index'], group['interpolated_population'], linewidth=2, label=str(sector_id))

ax.set_title('Interpolated population trajectories for representative sectors')
ax.set_xlabel('Epidemic week')
ax.set_ylabel('Population')
ax.set_xticks(week_positions)
ax.set_xticklabels(week_labels, rotation=30, ha='right')
ax.legend(title='Sector ID')
plt.tight_layout()

## Population change distribution and cross-section comparison

In [ ]:
first_week = epidemic_weeks[0]
middle_week = epidemic_weeks[len(epidemic_weeks) // 2]
last_week = epidemic_weeks[-1]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].hist(population_change, bins=40, color='#2A9D8F', edgecolor='white')
axes[0].axvline(population_change.mean(), color='black', linestyle='--', linewidth=1.5, label='Mean')
axes[0].set_title('Population change from 2010 to 2022')
axes[0].set_xlabel('2022 minus 2010 population')
axes[0].set_ylabel('Number of sectors')
axes[0].legend()

box_data = [
    interpolated_population[first_week],
    interpolated_population[middle_week],
    interpolated_population[last_week],
]
axes[1].boxplot(box_data, labels=['first week', 'middle week', 'last week'], showfliers=False)
axes[1].set_title('Cross-sector distribution at key epidemic weeks')
axes[1].set_ylabel('Population')

plt.tight_layout()

## Session save

In [ ]:
try:
    import dill

    session_path = PROJECT_ROOT / 'notebooks' / 'population_interpolation' / 'temp' / 'session.dill'
    session_path.parent.mkdir(parents=True, exist_ok=True)
    with open(session_path, 'wb') as file_handle:
        dill.dump_session(file_handle)
    print(f'Session saved to {session_path}')
except ImportError:
    print('dill is not installed; skipping session save.')